# TM-033 & TM34 & TM35— Final DistilBERT Prediction Notebook

This notebook implements the final single-model prediction pipeline required by TM-033:

```text
load final trained model → preprocess/tokenise test.csv → predict → write pred_bento.csv
```

Final selected model:

```text
DistilBERT | HF fine-tune
```

Expected trained checkpoint location:

```text
outputs/distilbert_checkpoints/distilbert-base-uncased/checkpoint-XXXX/
```

## 1. Setup project root

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working directory: {os.getcwd()}")

## 2. Imports from project scripts

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from transformers import (
    DistilBertForSequenceClassification,
    Trainer,
    DataCollatorWithPadding,
)

from src.config import (
    TEST_CSV_PATH,
    DISTILBERT_MODEL_NAME,
    DISTILBERT_CHECKPOINT_DIR,
    NUM_LABELS,
    LABEL2ID,
    ID2LABEL,
)

from src.distilbert_trainer import (
    load_tokenizer,
    predict_test_set,
)

from src.evaluate import save_submission

print("Imports OK")

## 3. Final prediction configuration

In [ ]:
OUTPUT_PATH = "outputs/pred_bento.csv"

# Usually leave this as None.
# If your colleague sends a checkpoint in another folder, paste the path here.
# Example:
# MANUAL_CHECKPOINT_PATH = "outputs/distilbert_checkpoints/distilbert-base-uncased/checkpoint-2862"
MANUAL_CHECKPOINT_PATH = None

checkpoint_root = Path(DISTILBERT_CHECKPOINT_DIR) / DISTILBERT_MODEL_NAME.replace("/", "_")

print(f"Test CSV path       : {TEST_CSV_PATH}")
print(f"Output CSV path     : {OUTPUT_PATH}")
print(f"Checkpoint root     : {checkpoint_root}")
print(f"Manual checkpoint   : {MANUAL_CHECKPOINT_PATH}")

## 4. Locate the trained DistilBERT checkpoint

This is the **load final trained model** part of the task.

The training script saves HuggingFace checkpoints using:

```python
save_strategy="epoch"
load_best_model_at_end=True
```

So this notebook expects one or more `checkpoint-*` folders under the configured checkpoint directory.

In [ ]:
def looks_like_hf_checkpoint(path: Path) -> bool:
    """Check whether a folder contains the minimum HuggingFace model files."""
    if not path.exists() or not path.is_dir():
        return False

    has_config = (path / "config.json").exists()
    has_weights = (
        (path / "model.safetensors").exists()
        or (path / "pytorch_model.bin").exists()
    )
    return has_config and has_weights


if MANUAL_CHECKPOINT_PATH is not None:
    final_checkpoint = Path(MANUAL_CHECKPOINT_PATH)

    if not looks_like_hf_checkpoint(final_checkpoint):
        raise FileNotFoundError(
            f"Manual checkpoint path is not a valid HuggingFace checkpoint:\n"
            f"{final_checkpoint}\n\n"
            "Expected files: config.json and model.safetensors or pytorch_model.bin."
        )

else:
    checkpoints = sorted(
        checkpoint_root.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )

    valid_checkpoints = [p for p in checkpoints if looks_like_hf_checkpoint(p)]

    if not valid_checkpoints:
        raise FileNotFoundError(
            f"No trained DistilBERT checkpoint found under:\n"
            f"{checkpoint_root}\n\n"
            "Copy the trained checkpoint folder into this location and run the notebook again.\n\n"
            "Expected example:\n"
            "outputs/distilbert_checkpoints/distilbert-base-uncased/checkpoint-XXXX/\n"
            "  ├── config.json\n"
            "  ├── model.safetensors or pytorch_model.bin\n"
            "  ├── trainer_state.json\n"
            "  └── training_args.bin\n"
        )

    # Use the last valid checkpoint by step number.
    final_checkpoint = valid_checkpoints[-1]

print(f"Selected checkpoint: {final_checkpoint}")

## 5. Load tokenizer and final trained model

We reuse the project helper `load_tokenizer()` from `src.distilbert_trainer`.

The trained model weights are loaded from the selected checkpoint.

In [ ]:
tokenizer = load_tokenizer()

model = DistilBertForSequenceClassification.from_pretrained(
    final_checkpoint,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Tokenizer loaded from project helper.")
print("Trained DistilBERT model loaded from checkpoint.")

## 6. Create inference trainer

The `Trainer` is only used here for prediction.  

In [ ]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

print("Inference Trainer ready.")

## 7. Predict on `test.csv`

We reuse `predict_test_set()` from `src.distilbert_trainer`.

Inside that project function, the pipeline is:

```text
load test.csv → HuggingFace Dataset → tokenizer(max_length=128) → trainer.predict → class ids
```

This keeps the final notebook consistent with the same tokenisation logic used by the DistilBERT training script.

In [ ]:
preds = predict_test_set(
    trainer=trainer,
    tokenizer=tokenizer,
    test_csv_path=TEST_CSV_PATH,
)

print(f"Generated predictions: {len(preds):,}")

## 8. Save final prediction CSV

We load `test.csv` only to recover the `id` column and then reuse the project helper `save_submission()`.

In [ ]:
test_df = pd.read_csv(TEST_CSV_PATH)

if "id" not in test_df.columns:
    test_df["id"] = range(len(test_df))

submission = save_submission(
    test_df=test_df,
    predictions=preds,
    output_path=OUTPUT_PATH,
)

submission.head()

## 9. Verify submission


In [ ]:
submission_verification = pd.read_csv("outputs/pred_bento.csv")

print(submission_verification.shape)

assert list(submission_verification.columns) == ["id", "label"]

assert submission_verification["label"].isin([0, 1, 2]).all()

print("CSV validation passed.")

In [ ]:
submission_verification.head()